# Uncertainty Quantification: Models That Know What They Don't Know

**Junior Design**

---

## The Question We Haven't Asked Yet

Every model we've built so far mainly answers one question: **"What is the prediction?"**

But there's an important question we've been ignoring: **"How confident should I be in that prediction?"**

Think about what happens when an engineer uses a model in practice:

- A model predicts that a reactor component will last 15 years. Should you replace it at 14 years? What if the model is uncertain — could the true answer be 8 years? Or 22 years?
- A model predicts that an air quality reading is "safe." Is it barely safe? Confidently safe? Should you issue a warning just in case?
- A model predicts that a building's energy consumption will be 450 MWh next month. Should the utility company plan for exactly that? Or should they budget for the possibility it could be 500?

The prediction alone isn't enough. **We need error bars.**

This is called **Uncertainty Quantification (UQ)** — and it's one of the most important topics in applied machine learning, especially in engineering and safety-critical applications.



---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.gaussian_process import GaussianProcessRegressor, GaussianProcessClassifier
from sklearn.gaussian_process.kernels import (
    RBF, ConstantKernel, Matern, WhiteKernel
)

from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.datasets import load_diabetes, load_breast_cancer, make_moons

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All imports successful!")

---

## Part 1: The Problem — Neural Networks Are Confidently Wrong

We'll create a simple scenario: fit a model to some data, then ask it to predict in a region **where it has never seen any data**.

A good model should say: "I don't know — I haven't seen data like this."

Let's see what a neural network actually does.

In [ ]:
# ============================================================
# Create data with a deliberate gap
# ============================================================
np.random.seed(42)

# Training data: observations on the LEFT and RIGHT, but nothing in the MIDDLE
X_left = np.random.uniform(0, 3, 25).reshape(-1, 1)
X_right = np.random.uniform(7, 10, 25).reshape(-1, 1)
X_train = np.vstack([X_left, X_right])

# True function: a sine wave with some noise
y_train = np.sin(X_train.ravel()) + 0.3 * np.random.randn(len(X_train))

# Where we'll ask for predictions (including the gap!)
X_plot = np.linspace(-1, 12, 300).reshape(-1, 1)
y_true = np.sin(X_plot.ravel())  # the true underlying function

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_train, y_train, c='black', s=40, zorder=5, label='Training data')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function (unknown to model)')
ax.axvspan(3, 7, alpha=0.1, color='red', label='GAP: no training data here!')
ax.axvspan(-1, 0, alpha=0.1, color='orange', label='EXTRAPOLATION: beyond training range')
ax.axvspan(10, 12, alpha=0.1, color='orange')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Our Setup: Training Data With a Gap', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("We have 50 training points, but NONE between x=3 and x=7.")
print("A model should be uncertain in the gap and beyond the edges.")
print("Let's see if our models know that.")

In [ ]:
# ============================================================
# Neural network: what does it predict in the gap?
# ============================================================
nn = Pipeline([
    ('scaler', StandardScaler()),
    ('nn', MLPRegressor(hidden_layer_sizes=(50, 25), activation='relu',
                         max_iter=2000, random_state=42))
])
nn.fit(X_train, y_train)
y_nn = nn.predict(X_plot)

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_train, y_train, c='black', s=40, zorder=5, label='Training data')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.plot(X_plot, y_nn, 'red', linewidth=2.5, label='Neural network prediction')
ax.axvspan(3, 7, alpha=0.07, color='red')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Neural Network: Confident Predictions Everywhere', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Look at the gap region (x = 3 to 7) and the extrapolation regions.")
print("The neural network draws a confident line through areas where it has")
print("NO data at all. There are no error bars, no warning, no hesitation.")
print("\nIf you were an engineer relying on this prediction, you'd have no idea")
print("that the model is essentially making things up in those regions.")

In [ ]:
# ============================================================
# The problem gets worse: different random seeds, different guesses
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_train, y_train, c='black', s=40, zorder=5, label='Training data')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')

for seed in range(10):
    nn_i = Pipeline([
        ('scaler', StandardScaler()),
        ('nn', MLPRegressor(hidden_layer_sizes=(50, 25), activation='relu',
                             max_iter=2000, random_state=seed))
    ])
    nn_i.fit(X_train, y_train)
    y_i = nn_i.predict(X_plot)
    ax.plot(X_plot, y_i, alpha=0.4, linewidth=1)

ax.axvspan(3, 7, alpha=0.07, color='red')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('10 Neural Networks (Different Random Seeds) — Same Data', 
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

print("Each neural network gives a DIFFERENT prediction in the gap and edges.")
print("Near the training data, they mostly agree (they're all learning the same pattern).")
print("In the gap? They wildly disagree — but each one acts 100% confident.")
print("\nThis spread IS the uncertainty — but a single neural network hides it from you.")

### The Fundamental Problem

Standard neural networks (and linear regression, decision trees, SVMs) produce **point predictions** — a single number with no uncertainty attached.

They have **no mechanism** for saying:
- "I've seen lots of data like this — I'm confident" (near training data)
- "I've never seen anything like this — don't trust me" (in gaps or extrapolation)

We need a model that **automatically quantifies its own uncertainty**.

Enter: **Gaussian Processes.**

---

## Part 2: What Is a Gaussian Process?

### The Intuition (No Math Yet)

Think of a GP this way:

**A neural network learns ONE function** that fits the data. It gives you a single curve and says "this is the answer."

**A Gaussian Process considers ALL possible functions** that could fit the data, and tells you how probable each one is.

Where there's lots of training data, almost all plausible functions pass through roughly the same place → **low uncertainty**.

Where there's no training data, the plausible functions spread out wildly → **high uncertainty**.

The GP summarizes this at each point with:
- A **mean** (the best-guess prediction — average of all plausible functions)
- A **standard deviation** (how much the plausible functions disagree)

Let's see this in action.

In [ ]:
# ============================================================
# Gaussian Process on the same data — now with uncertainty
# ============================================================
kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,  # try multiple starting points for kernel optimization
    random_state=42
)
gp.fit(X_train, y_train)

# The key: predict returns both mean and uncertainty
y_mean, y_std = gp.predict(X_plot, return_std=True)

fig, ax = plt.subplots(figsize=(12, 6))

# Confidence bands: mean ± 1σ and ± 2σ
ax.fill_between(X_plot.ravel(), y_mean - 2*y_std, y_mean + 2*y_std, 
                alpha=0.15, color='steelblue', label='95% confidence (±2σ)')
ax.fill_between(X_plot.ravel(), y_mean - y_std, y_mean + y_std, 
                alpha=0.25, color='steelblue', label='68% confidence (±1σ)')

# Mean prediction
ax.plot(X_plot, y_mean, 'steelblue', linewidth=2.5, label='GP mean prediction')

# True function and data
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.scatter(X_train, y_train, c='black', s=40, zorder=5, label='Training data')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gaussian Process: Predictions WITH Uncertainty', fontweight='bold', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

print("This is what uncertainty quantification looks like.")
print("\nNear training data: bands are NARROW (model is confident)")
print("In the gap (x=3-7): bands WIDEN (model admits it's uncertain)")
print("Beyond the edges:   bands WIDEN (model knows it's extrapolating)")
print("\nThe true function (dashed line) mostly falls WITHIN the bands.")
print("The GP doesn't just predict — it tells you how much to trust it.")

In [ ]:
# ============================================================
# Side-by-side: Neural Network vs Gaussian Process
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Neural Network ---
ax = axes[0]
ax.scatter(X_train, y_train, c='black', s=40, zorder=5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.plot(X_plot, y_nn, 'red', linewidth=2.5)
ax.axvspan(3, 7, alpha=0.07, color='red')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Neural Network\n"The answer is 0.3" (no uncertainty)', 
             fontweight='bold', fontsize=13, color='red')
ax.set_ylim(-2.5, 3)

# --- Gaussian Process ---
ax = axes[1]
ax.fill_between(X_plot.ravel(), y_mean - 2*y_std, y_mean + 2*y_std, 
                alpha=0.15, color='steelblue')
ax.fill_between(X_plot.ravel(), y_mean - y_std, y_mean + y_std, 
                alpha=0.25, color='steelblue')
ax.plot(X_plot, y_mean, 'steelblue', linewidth=2.5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.scatter(X_train, y_train, c='black', s=40, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gaussian Process\n"The answer is 0.1 ± 0.8" (with uncertainty)', 
             fontweight='bold', fontsize=13, color='steelblue')
ax.set_ylim(-2.5, 3)

plt.suptitle(' Point Prediction vs. Uncertainty-Aware Prediction', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### How Does a GP Actually Work?

The math behind GPs is pretty slick but we'll focus on the intuition. Here's the key idea:

**A GP assumes that nearby inputs produce similar outputs.**

If you know that $f(3.0) = 1.2$, then $f(3.1)$ is probably close to 1.2 as well. $f(3.01)$ is almost certainly close. $f(50)$? No idea — it's too far away to infer anything.

This notion of "nearby" is controlled by a **kernel function** (also called a covariance function), which measures the similarity between any two input points. The kernel is the heart of the GP.

**The GP prediction at a new point $x_*$:**

1. Look at how similar $x_*$ is to every training point (using the kernel)
2. Training points that are very similar to $x_*$ get a strong vote on the prediction
3. Training points that are far away get almost no vote
4. If $x_*$ is far from ALL training points, no one gets a strong vote → high uncertainty

Formally, the prediction is a weighted average of training outputs, where the weights come from the kernel. The uncertainty comes from how much "support" the prediction has from nearby training data.

In [ ]:
# ============================================================
# Visualize: how the GP "sees" similarity between points
# ============================================================
# Pick three query points: one near data, one in the gap, one far out
query_points = [1.5, 5.0, 14.0]
query_labels = ['Near training data (x=1.5)', 'In the gap (x=5.0)', 'Far out (x=14.0)']
query_colors = ['green', 'orange', 'red']

# Use a simple RBF kernel to show similarity
length_scale = gp.kernel_.get_params()['k1__k2__length_scale']

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for ax, qp, label, color in zip(axes, query_points, query_labels, query_colors):
    # Compute similarity (kernel value) between query point and all training points
    distances = np.abs(X_train.ravel() - qp)
    similarities = np.exp(-0.5 * (distances / length_scale)**2)
    
    ax.bar(range(len(X_train)), similarities, color=color, alpha=0.6, edgecolor='black')
    ax.set_xlabel('Training point index')
    ax.set_ylabel('Kernel similarity')
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.1)
    
    # Annotate
    max_sim = similarities.max()
    n_relevant = (similarities > 0.1).sum()
    ax.text(0.95, 0.95, f'Max similarity: {max_sim:.2f}\nRelevant points: {n_relevant}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('How the GP Weighs Training Points for Each Prediction', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Near data (x=1.5):  Many training points are similar → strong evidence → low uncertainty")
print("In the gap (x=5.0): No training points are similar → weak evidence → high uncertainty")
print("Far out (x=14.0):   No training points are similar → no evidence → highest uncertainty")

---

## Part 3: GP Kernels — Controlling What the Model Assumes

The **kernel** (covariance function) is the most important choice you make when using a GP. It encodes your assumptions about the function you're trying to learn:

- How smooth is the function?
- How quickly does similarity decay with distance?
- Is there noise in the measurements?

### Common Kernels

| Kernel | What It Assumes | When to Use |
|---|---|---|
| **RBF** (Radial Basis Function) | Function is infinitely smooth | Default choice, works well for many problems |
| **Matérn** | Function is smooth but not infinitely so | More realistic for physical processes |
| **WhiteKernel** | There is observation noise | Almost always added to account for measurement error |
| **ConstantKernel** | Scales the overall variance | Combined with other kernels to control amplitude |

In practice, you combine kernels: `ConstantKernel * RBF + WhiteKernel` means "the function is smooth (RBF), with adjustable amplitude (Constant), and the observations have noise (White)."

The GP **automatically tunes** kernel parameters (like the length scale and noise level) by maximizing the likelihood of the training data.

In [ ]:
# ============================================================
# Visualize: what the length scale parameter does
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

length_scales = [0.3, 1.0, 5.0]
ls_labels = ['Short (0.3)\nWiggly, local', 'Medium (1.0)\nBalanced', 'Long (5.0)\nSmooth, global']

for ax, ls, label in zip(axes, length_scales, ls_labels):
    kernel_i = ConstantKernel(1.0) * RBF(length_scale=ls) + WhiteKernel(noise_level=0.1)
    # Fix hyperparameters (don't optimize) to show the effect
    gp_i = GaussianProcessRegressor(kernel=kernel_i, optimizer=None, random_state=42)
    gp_i.fit(X_train, y_train)
    y_mean_i, y_std_i = gp_i.predict(X_plot, return_std=True)
    
    ax.fill_between(X_plot.ravel(), y_mean_i - 2*y_std_i, y_mean_i + 2*y_std_i, 
                    alpha=0.15, color='steelblue')
    ax.plot(X_plot, y_mean_i, 'steelblue', linewidth=2)
    ax.plot(X_plot, y_true, 'k--', alpha=0.3)
    ax.scatter(X_train, y_train, c='black', s=30, zorder=5)
    ax.set_title(f'Length scale = {label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylim(-3, 3)

plt.suptitle('Effect of Length Scale: How Far Does Each Point\'s Influence Reach?', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Short length scale: Each point only influences very nearby predictions.")
print("  → Wiggly fit, rapid uncertainty growth away from data.")
print("\nLong length scale: Each point influences a wide region.")
print("  → Smooth fit, uncertainty grows slowly.")
print("\nIn practice, the GP optimizes this automatically from the data.")

In [ ]:
# ============================================================
# Compare kernels: RBF vs Matérn
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

kernels_compare = [
    ('RBF (infinitely smooth)', ConstantKernel(1.0) * RBF(1.0) + WhiteKernel(0.1)),
    ('Matérn ν=2.5 (realistic smoothness)', ConstantKernel(1.0) * Matern(1.0, nu=2.5) + WhiteKernel(0.1)),
]

for ax, (name, kernel_i) in zip(axes, kernels_compare):
    gp_i = GaussianProcessRegressor(kernel=kernel_i, n_restarts_optimizer=10, random_state=42)
    gp_i.fit(X_train, y_train)
    y_m, y_s = gp_i.predict(X_plot, return_std=True)
    
    ax.fill_between(X_plot.ravel(), y_m - 2*y_s, y_m + 2*y_s, alpha=0.15, color='steelblue')
    ax.fill_between(X_plot.ravel(), y_m - y_s, y_m + y_s, alpha=0.25, color='steelblue')
    ax.plot(X_plot, y_mean, 'steelblue', linewidth=2)
    ax.plot(X_plot, y_true, 'k--', alpha=0.3)
    ax.scatter(X_train, y_train, c='black', s=30, zorder=5)
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('x')
    ax.set_ylim(-3, 3)

plt.suptitle('Kernel Comparison: Smoothness Assumptions', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("For most engineering applications, Matérn (ν=2.5) is a good default.")
print("It assumes the function is smooth but not unrealistically so.")
print("RBF can over-smooth and be overconfident between clusters of points.")

In [ ]:
# ============================================================
# What happens as we add more training data?
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

data_sizes = [5, 10, 20, 40, 60, 100]

for ax, n in zip(axes.flat, data_sizes):
    np.random.seed(42)
    X_sub = np.random.uniform(0, 10, n).reshape(-1, 1)
    y_sub = np.sin(X_sub.ravel()) + 0.3 * np.random.randn(n)
    
    kernel_i = ConstantKernel(1.0) * Matern(1.0, nu=2.5) + WhiteKernel(0.1)
    gp_i = GaussianProcessRegressor(kernel=kernel_i, n_restarts_optimizer=5, random_state=42)
    gp_i.fit(X_sub, y_sub)
    y_m, y_s = gp_i.predict(X_plot, return_std=True)
    
    ax.fill_between(X_plot.ravel(), y_m - 2*y_s, y_m + 2*y_s, alpha=0.15, color='steelblue')
    ax.plot(X_plot, y_m, 'steelblue', linewidth=2)
    ax.plot(X_plot, y_true, 'k--', alpha=0.3)
    ax.scatter(X_sub, y_sub, c='black', s=30, zorder=5)
    ax.set_title(f'n = {n} training points', fontweight='bold')
    ax.set_ylim(-3, 3)

plt.suptitle('GP Uncertainty Shrinks as You Add More Data', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("With 5 points: Wide uncertainty everywhere.")
print("With 100 points: Tight bands — the GP is confident because")
print("  it has data throughout the input space.")
print("\nThis is exactly what you'd want: more data → more confidence.")

---

## Part 4: GP Uncertainty on Real Data

The 1D examples are visually clear, but real data has many features. Let's apply GPs to the **diabetes dataset** — a real medical regression dataset built into sklearn. The task is predicting disease progression from 10 baseline measurements (age, BMI, blood pressure, blood serum measurements, etc.).

We can't visualize uncertainty bands in 10 dimensions, but we can ask: **are the GP's uncertainty estimates trustable?**

In [ ]:
# ============================================================
# Load and explore the diabetes dataset
# ============================================================
diabetes = load_diabetes()
X_diab = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diab = diabetes.target

print(f"Dataset: {X_diab.shape[0]} patients, {X_diab.shape[1]} features")
print(f"Target: disease progression score ({y_diab.min():.0f} to {y_diab.max():.0f})")
print(f"\nFeatures:")
for col in X_diab.columns:
    print(f"  {col}: {X_diab[col].min():.4f} to {X_diab[col].max():.4f}")

In [ ]:
# ============================================================
# Train GP and neural network on diabetes data
# ============================================================
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    diabetes.data, y_diab, test_size=0.2, random_state=42
)

# Scale features (important for both GP and NN)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_d)
X_test_sc = scaler.transform(X_test_d)

# --- Gaussian Process ---
kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1.0)
gp_diab = GaussianProcessRegressor(
    kernel=kernel, n_restarts_optimizer=10, random_state=42, alpha=1e-6
)
gp_diab.fit(X_train_sc, y_train_d)
y_pred_gp, y_std_gp = gp_diab.predict(X_test_sc, return_std=True)

# --- Neural Network ---
nn_diab = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
nn_diab.fit(X_train_sc, y_train_d)
y_pred_nn = nn_diab.predict(X_test_sc)

# --- Linear Regression ---
lr_diab = LinearRegression()
lr_diab.fit(X_train_sc, y_train_d)
y_pred_lr = lr_diab.predict(X_test_sc)

print("Model Performance (R² on test set):")
print(f"  Linear Regression:  {r2_score(y_test_d, y_pred_lr):.4f}")
print(f"  Neural Network:     {r2_score(y_test_d, y_pred_nn):.4f}")
print(f"  Gaussian Process:   {r2_score(y_test_d, y_pred_gp):.4f}")
print(f"\nBut the GP gives us something the others don't: per-prediction uncertainty.")

In [ ]:
# ============================================================
# Are the GP's uncertainty estimates calibrated?
# ============================================================
# If the uncertainty is well-calibrated:
#   ~68% of true values should fall within ±1σ of the prediction
#   ~95% of true values should fall within ±2σ

within_1std = np.mean(np.abs(y_test_d - y_pred_gp) < 1 * y_std_gp)
within_2std = np.mean(np.abs(y_test_d - y_pred_gp) < 2 * y_std_gp)
within_3std = np.mean(np.abs(y_test_d - y_pred_gp) < 3 * y_std_gp)

print("GP Calibration Check:")
print(f"  Within ±1σ: {within_1std:.1%}  (ideal: ~68%)")
print(f"  Within ±2σ: {within_2std:.1%}  (ideal: ~95%)")
print(f"  Within ±3σ: {within_3std:.1%}  (ideal: ~99.7%)")
print(f"\nIf these roughly match, the GP's uncertainty estimates are trustworthy.")

In [ ]:
# ============================================================
# Visualize: predictions with error bars vs point predictions
# ============================================================
# Sort test points by predicted value for cleaner visualization
sort_idx = np.argsort(y_pred_gp)
patient_idx = np.arange(len(y_test_d))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Neural Network: point predictions only ---
ax = axes[0]
ax.scatter(patient_idx, y_test_d[sort_idx], c='black', s=20, 
           label='True values', zorder=5)
ax.scatter(patient_idx, y_pred_nn[sort_idx], c='red', s=20, marker='x',
           label='NN predictions', alpha=0.7)
ax.set_xlabel('Patient (sorted by GP prediction)')
ax.set_ylabel('Disease Progression')
ax.set_title('Neural Network\n(point predictions — no uncertainty)', 
             fontweight='bold', fontsize=12, color='red')
ax.legend(fontsize=10)

# --- Gaussian Process: predictions WITH error bars ---
ax = axes[1]
ax.scatter(patient_idx, y_test_d[sort_idx], c='black', s=20, 
           label='True values', zorder=5)
ax.errorbar(patient_idx, y_pred_gp[sort_idx], yerr=2*y_std_gp[sort_idx],
            fmt='o', markersize=3, color='steelblue', alpha=0.6, capsize=2,
            label='GP prediction ± 2σ')
ax.set_xlabel('Patient (sorted by GP prediction)')
ax.set_ylabel('Disease Progression')
ax.set_title('Gaussian Process\n(predictions WITH uncertainty)', 
             fontweight='bold', fontsize=12, color='steelblue')
ax.legend(fontsize=10)

plt.suptitle('Same Patients, Same Predictions — But Only One Model Shows Confidence', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("The neural network gives you a dot. Take it or leave it.")
print("The GP gives you a dot PLUS error bars: 'my best guess is 150, but")
print("it could reasonably be anywhere from 95 to 205.'")
print("\nWhich would you rather have if you were making a medical decision?")

In [ ]:
# ============================================================
# Predicted vs Actual with uncertainty bands
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# NN: predicted vs actual (no uncertainty)
ax = axes[0]
ax.scatter(y_test_d, y_pred_nn, c='red', alpha=0.6, edgecolors='black', s=40)
ax.plot([0, 350], [0, 350], 'k--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual Disease Progression')
ax.set_ylabel('Predicted')
ax.set_title('Neural Network', fontweight='bold', color='red')
ax.legend()

# GP: predicted vs actual with error bars
ax = axes[1]
ax.errorbar(y_test_d, y_pred_gp, yerr=2*y_std_gp, fmt='o', 
            color='steelblue', alpha=0.5, capsize=2, markersize=4,
            markeredgecolor='black', markeredgewidth=0.5)
ax.plot([0, 350], [0, 350], 'k--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual Disease Progression')
ax.set_ylabel('Predicted')
ax.set_title('Gaussian Process (with ±2σ bars)', fontweight='bold', color='steelblue')
ax.legend()

plt.suptitle('Predicted vs Actual — Diabetes Dataset', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()